# Phase 5 Pipeline B — Recoloration smoke-test

Tests Grounding DINO + SAM2 + HSV recoloration on ~36 images from the source pool (3 per object × 12 objects). Inspect the output grids visually before running the full recoloration pipeline.

**Runtime:** ~15-25 min on A100, ~30-40 on L4 (model loading dominates first call).

**Output:** side-by-side grids in `/content/recolor_smoke/grids/` showing [original | mask overlay | recolored].

## 1. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/GabrielmAlves/dgm-2026.1.git"
BRANCH = "feature/generate_datasets"

import os, subprocess
REPO_DIR = "/content/binding-research"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR

## 2. Install dependencies

Heavy step: SAM2 has its own package and the Hiera checkpoint is ~2.5GB. First run takes 5-10 min.

In [ ]:
!pip install -q uv
!uv pip install -e . --system

!pip install -q git+https://github.com/facebookresearch/sam2.git

!pip install -q transformers accelerate

## 3. HF login

Grounding DINO checkpoint downloads automatically once authenticated.

In [ ]:
from huggingface_hub import login
login()

## 4. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
assert vram >= 16, 'Switch to A100 or L4 (SAM2-large + Grounding DINO needs ~8-10 GB)'

## 5. Symlink the source pool from Drive

The pool CSV has absolute paths from the Windows machine; we need to redirect them to the Drive-mounted candidates directories. The script reads the CSV as-is, so we recreate the parent paths as symlinks pointing at the Drive.

In [ ]:
import csv
from pathlib import Path

POOL_CSV = Path('/content/drive/MyDrive/binding-research/exp5_pool/source_pool.csv')
assert POOL_CSV.exists(), f'Upload source_pool.csv to {POOL_CSV} first'

DRIVE_ROOT = Path('/content/drive/MyDrive/binding-research/finetuning')

out_csv = Path('/content/source_pool_colab.csv')
with POOL_CSV.open(newline='') as fin, out_csv.open('w', newline='') as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=reader.fieldnames)
    writer.writeheader()
    for row in reader:
        if row['source_collection'] == 'control':
            root = DRIVE_ROOT / 'control_candidates'
        else:
            root = DRIVE_ROOT / 'canonical_extra_candidates'
        win_path = row['path'].replace('\\', '/')
        parts = win_path.split('/')
        rel = '/'.join(parts[-3:])  
        row['path'] = str(root / rel)
        writer.writerow(row)
print(f'Wrote rewritten pool: {out_csv}')

with out_csv.open() as f:
    print(next(csv.DictReader(f)))

## 6. Run the smoke-test

In [ ]:
!python experiments/exp5_recolor_smoke.py \
    --pool /content/source_pool_colab.csv \
    --out /content/recolor_smoke \
    --n-per-object 3

## 7. Inspect a few grids inline

In [ ]:
from PIL import Image
import os
from IPython.display import display

GRIDS = '/content/recolor_smoke/grids'
files = sorted(os.listdir(GRIDS))[:6]
for fn in files:
    print(fn)
    display(Image.open(os.path.join(GRIDS, fn)))

## 8. Save grids and summary to Drive

In [ ]:
import shutil
DEST = '/content/drive/MyDrive/binding-research/exp5_pool/recolor_smoke'
shutil.copytree('/content/recolor_smoke', DEST, dirs_exist_ok=True)
print(f'Saved to {DEST}')